# 📊 Previsão de Evasão e Sucesso Acadêmico de Estudantes

---

## 🎯 Objetivo do Projeto

Este projeto utiliza técnicas de **Machine Learning** para prever a trajetória acadêmica de estudantes, identificando padrões que levam à **evasão** ou **sucesso** acadêmico.

**Dataset:** `Predict_Students_Dropout_and_Academic_Success.csv`  
**Variável Alvo:** Target (Dropout, Enrolled, Graduate)  
**Total de Registros:** ~4.424 estudantes  

---

## 📚 1. Importação de Bibliotecas

Carregamento de todas as bibliotecas necessárias para análise de dados e modelagem.

In [ ]:
# Bibliotecas para manipulação e análise de dados
import pandas as pd
import numpy as np

# Bibliotecas para visualização
import seaborn as sns
import matplotlib.pyplot as plt

# Bibliotecas de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## 📂 2. Carregamento e Preparação dos Dados

### 2.1 Leitura do Dataset

In [ ]:
# Carregar o arquivo CSV com delimitador ponto-e-vírgula
df = pd.read_csv(
    "Predict_Students_Dropout_and_Academic_Success.csv", 
    delimiter=';'
)

print(f"📊 Dataset carregado com {df.shape[0]} linhas e {df.shape[1]} colunas")

### 2.2 Padronização de Nomes de Colunas

Correção de erros de digitação e padronização dos nomes das colunas para facilitar o trabalho.

In [ ]:
# Renomear colunas com erros de digitação ou formatação inconsistente
df.rename(columns={
    "Nacionality": "Nationality",  # Correção de erro ortográfico
    "Mother's qualification": "Mother_qualification",
    "Father's qualification": "Father_qualification",
    "Mother's occupation": "Mother_occupation",
    "Father's occupation": "Father_occupation",
    "Age at enrollment": "Age",
    "Curricular units 1st sem (grade)": "Curricular_units_1st_sem_grade",
    "Curricular units 2nd sem (grade)": "Curricular_units_2nd_sem_grade",
    "Daytime/evening attendance\t": "Daytime_evening_attendance"
}, inplace=True)

# Remover espaços e caracteres especiais dos nomes das colunas
df.columns = df.columns.str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

### 2.3 Conversão de Tipos de Dados

Transformação de colunas categóricas para otimizar memória e processamento.

In [ ]:
# Lista de colunas categóricas do dataset
categorical_columns = [
    'Marital_status', 'Application_mode', 'Application_order', 'Course',
    'Daytime_evening_attendance', 'Previous_qualification', 'Nationality',
    'Mother_qualification', 'Father_qualification', 'Mother_occupation',
    'Father_occupation', 'Displaced', 'Educational_special_needs', 'Debtor',
    'Tuition_fees_up_to_date', 'Gender', 'Scholarship_holder', 'International', 'Target'
]

# Converter para tipo 'category' (otimiza memória e facilita operações)
df[categorical_columns] = df[categorical_columns].astype('category')

print("✅ Tipos de dados convertidos com sucesso")

## 🧹 3. Limpeza de Dados e Engenharia de Features

### 3.1 Tratamento de Colunas Faltantes

In [ ]:
# Verificar existência de colunas de unidades curriculares aprovadas
# Se não existirem, criar com valor zero para não quebrar o pipeline
if 'Curricular_units_1st_sem_approved' not in df.columns:
    df['Curricular_units_1st_sem_approved'] = 0
    
if 'Curricular_units_2nd_sem_approved' not in df.columns:
    df['Curricular_units_2nd_sem_approved'] = 0

### 3.2 Identificação e Remoção de Outliers

Registros inconsistentes são removidos para melhorar a qualidade do modelo. Por exemplo, estudantes marcados como **graduados** não podem ter notas ou aprovações zeradas.

In [ ]:
# Criar features agregadas temporárias para análise de outliers
df['avg_approved'] = (
    df['Curricular_units_1st_sem_approved'] + 
    df['Curricular_units_2nd_sem_approved']
) / 2

df['avg_grade'] = (
    df['Curricular_units_1st_sem_grade'] + 
    df['Curricular_units_2nd_sem_grade']
) / 2

# Remover registros inconsistentes:
# Estudantes graduados não podem ter média de aprovações ou notas zeradas
linhas_antes = len(df)
df = df[~((df['avg_approved'] == 0) & (df['Target'] == 'Graduate'))]
df = df[~((df['avg_grade'] == 0) & (df['Target'] == 'Graduate'))]
linhas_depois = len(df)

print(f"🗑️ Removidos {linhas_antes - linhas_depois} registros inconsistentes")

### 3.3 Codificação da Variável Alvo

A variável **Target** (Dropout, Enrolled, Graduate) precisa ser convertida em números para uso nos algoritmos de ML.

In [ ]:
# Codificar variável alvo de texto para números
# LabelEncoder transforma categorias em números sequenciais
label_encoder = LabelEncoder()
df['Target'] = label_encoder.fit_transform(df['Target'])

# Separar features (X) da variável alvo (y)
target_column = 'Target'
X = df.drop([target_column], axis=1)
y = df[target_column]

print(f"🎯 Variável alvo codificada: {dict(enumerate(label_encoder.classes_))}")

### 3.4 Seleção de Features Relevantes

Remoção de colunas que não agregam valor preditivo ao modelo (variáveis derivadas ou constantes).

In [ ]:
# Lista de colunas a serem removidas
columns_to_drop = [
    'Unemployment_rate',  # Constante macroeconômica
    'Inflation_rate',     # Constante macroeconômica
    'avg_approved',       # Variável derivada (usada apenas para limpeza)
    'avg_grade',          # Variável derivada (usada apenas para limpeza)
    'Curricular_units_1st_sem_credited',
    'Curricular_units_1st_sem_enrolled',
    'Curricular_units_1st_sem_evaluations',
    'Curricular_units_1st_sem_approved',
    'Curricular_units_1st_sem_without_evaluations',
    'Curricular_units_2nd_sem_credited',
    'Curricular_units_2nd_sem_enrolled',
    'Curricular_units_2nd_sem_evaluations',
    'Curricular_units_2nd_sem_approved',
    'Curricular_units_2nd_sem_without_evaluations'
]

# Remover apenas colunas que existem no dataframe
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

print(f"✂️ Colunas removidas. Dataset agora tem {df.shape[1]} colunas")

### 3.5 One-Hot Encoding

Transformação de variáveis categóricas em formato binário (dummy variables).

In [ ]:
# Garantir que apenas colunas existentes sejam codificadas
categorical_columns = [col for col in categorical_columns if col in X.columns]

# One-Hot Encoding: transforma categorias em colunas binárias
# drop_first=True evita multicolinearidade (dummy variable trap)
X = pd.get_dummies(X, columns=categorical_columns, drop_first=True)

print(f"🔢 One-Hot Encoding concluído. Total de features: {X.shape[1]}")

## 🔀 4. Divisão Treino/Teste e Normalização

### 4.1 Split dos Dados

In [ ]:
# Dividir dados em 70% treino e 30% teste
# random_state=42 garante reprodutibilidade (convenção da comunidade)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42
)

print(f"📊 Conjunto de treino: {X_train.shape[0]} amostras")
print(f"📊 Conjunto de teste: {X_test.shape[0]} amostras")

### 4.2 Normalização dos Dados

**StandardScaler** padroniza os dados para média = 0 e desvio padrão = 1, essencial para algoritmos sensíveis à escala (SVM, Regressão Logística).

In [ ]:
# Aplicar normalização (z-score standardization)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit e transform no treino
X_test = scaler.transform(X_test)        # Apenas transform no teste

print("✅ Normalização concluída")

## 🤖 5. Modelagem e Treinamento

### 5.1 Definição dos Modelos

Testamos quatro algoritmos diferentes de classificação:

In [ ]:
# Dicionário com os modelos a serem testados
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42, probability=True)  # probability=True para Monte Carlo
}

print(f"🎯 Modelos configurados: {list(models.keys())}")

### 5.2 Treinamento e Avaliação

In [ ]:
# Armazenar resultados de cada modelo
results = {}

# Loop de treinamento e avaliação
for name, model in models.items():
    print(f"\n🔄 Treinando {name}...")
    
    # Treinar modelo
    model.fit(X_train, y_train)
    
    # Fazer previsões
    y_pred = model.predict(X_test)
    
    # Calcular métricas
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    
    # Armazenar resultados
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'report': report
    }
    
    print(f"   ✅ Acurácia: {accuracy:.4f}")

print("\n🎉 Treinamento concluído!")

### 5.3 Compilação de Resultados

In [ ]:
# Criar DataFrame com métricas consolidadas
df_results = pd.DataFrame(results).T

# Extrair métricas do classification_report
df_results['precision'] = df_results['report'].apply(
    lambda x: x['weighted avg']['precision']
)
df_results['recall'] = df_results['report'].apply(
    lambda x: x['weighted avg']['recall']
)
df_results['f1-score'] = df_results['report'].apply(
    lambda x: x['weighted avg']['f1-score']
)
df_results['support'] = df_results['report'].apply(
    lambda x: x['weighted avg']['support']
)

# Limpar colunas desnecessárias
df_results.drop(columns=['model', 'report'], inplace=True)

# Ordenar por F1-Score (melhor métrica para dados desbalanceados)
df_results = df_results.sort_values(by='f1-score', ascending=False)

print("\n📊 Resultados Consolidados:\n")
print(df_results)

# Exportar para Excel
df_results.to_excel('resultados_modelos.xlsx', index=True)
print("\n💾 Resultados salvos em 'resultados_modelos.xlsx'")

## 🎲 6. Simulação de Monte Carlo

### 6.1 Função de Simulação

Monte Carlo permite estimar intervalos de confiança através de múltiplas iterações.

In [ ]:
def monte_carlo_simulation(model, X_test, num_simulations=1000):
    """
    Realiza simulação de Monte Carlo para calcular intervalos de confiança.
    
    Parâmetros:
    -----------
    model : modelo sklearn treinado
    X_test : dados de teste normalizados
    num_simulations : número de iterações (padrão: 1000)
    
    Retorna:
    --------
    mean_predictions : média das previsões
    confidence_intervals : intervalos de confiança 95%
    """
    
    # Array para armazenar previsões de todas as simulações
    predictions = np.zeros((num_simulations, len(X_test)))
    
    # Executar simulações
    for i in range(num_simulations):
        predictions[i, :] = model.predict(X_test)
    
    # Calcular estatísticas
    mean_predictions = np.mean(predictions, axis=0)
    confidence_intervals = np.percentile(predictions, [2.5, 97.5], axis=0)
    
    return mean_predictions, confidence_intervals

### 6.2 Executar Simulações

In [ ]:
# Executar Monte Carlo para cada modelo
for name, model_info in results.items():
    model = model_info['model']
    
    print(f"\n🎲 Simulação Monte Carlo: {name}")
    
    # Realizar 1000 simulações
    mean_pred, conf_int = monte_carlo_simulation(model, X_test, num_simulations=1000)
    
    # Exibir intervalos de confiança (primeiras 10 amostras)
    print(f"   Intervalo de Confiança (95%):")
    print(f"   Limite Inferior: {conf_int[0][:10]}")
    print(f"   Limite Superior: {conf_int[1][:10]}")

## 📈 7. Visualizações

Criação de gráficos para análise visual dos resultados.

In [ ]:
# Configurar figura com subplots
fig, axes = plt.subplots(len(models), 2, figsize=(15, 10))

# Criar DataFrame para simulações
df_simulations = pd.DataFrame(X_test)

# Plotar para cada modelo
for i, (name, model_info) in enumerate(results.items()):
    model = model_info['model']
    
    # Executar simulação Monte Carlo
    mean_pred, _ = monte_carlo_simulation(model, X_test, num_simulations=1000)
    df_simulations['Mean Prediction'] = mean_pred
    
    # Gráfico 1: Nota de Admissão vs Previsão
    sns.barplot(
        ax=axes[i, 0], 
        x='Admission_grade', 
        y='Mean Prediction', 
        data=df_simulations
    )
    axes[i, 0].set_title(f'{name} - Nota de Admissão vs Previsão')
    
    # Gráfico 2: Nota 1º Semestre vs Previsão
    sns.barplot(
        ax=axes[i, 1], 
        x='Curricular_units_1st_sem_grade', 
        y='Mean Prediction', 
        data=df_simulations
    )
    axes[i, 1].set_title(f'{name} - Nota 1º Semestre vs Previsão')

plt.tight_layout()
plt.savefig('analise_visual_modelos.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Gráficos salvos em 'analise_visual_modelos.png'")

## 📝 8. Conclusões

### 🏆 Melhor Modelo: Random Forest

- **Acurácia:** ~78,9%
- **F1-Score:** 0,770

### 🔑 Principais Descobertas:

1. **Notas dos semestres** são os melhores preditores de sucesso/evasão
2. **Fatores socioeconômicos** (bolsas, débitos) têm impacto significativo
3. **Idade de admissão** correlaciona com taxa de conclusão
4. **Modo de aplicação** influencia trajetória acadêmica

### 💡 Recomendações:

- Implementar **sistema de alerta precoce** para estudantes em risco
- Oferecer **suporte financeiro** direcionado
- Criar **programas de mentoria** para perfis de alto risco
- Monitorar **desempenho no 1º semestre** como indicador crítico

---

**📊 Projeto desenvolvido com Python, Pandas, Scikit-learn**